# Evaluate registered BioASQ retrieval datasets

This notebook registers ranking pipelines, an independent evaluation definition, and runtime-only settings. It evaluates BM25 and Qwen3-Embedding-0.6B on every latest dataset below `DATASETS_ROOT`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} pull --ff-only
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets sentence-transformers rank-bm25


In [ ]:
import logging

import pandas as pd
import torch
from IPython.display import display

from src.evaluate import (
    RuntimeConfig,
    evaluate,
    register_evaluation,
    register_pipeline,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("retrieval-test")


## Configuration

`PipelineDefinition` contains only settings that can change rankings. Metric cutoffs are registered separately as an `EvaluationDefinition`. Batch size, corpus scan size, progress output, and device are runtime-only and never change either identity.

In [ ]:
DATASETS_ROOT = "/content/drive/MyDrive/Retreaval/data"
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"

METRIC_CONFIG = {
    "mrr_at_k": (10,),
    "ndcg_at_k": (10,),
    "accuracy_at_k": (1, 3, 5, 10, 100),
    "precision_recall_at_k": (1, 3, 5, 10, 100),
    "map_at_k": (100,),
}

runtime = RuntimeConfig(
    batch_size=32,
    corpus_scan_size=10_000,
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
evaluation_id = register_evaluation(
    METRIC_CONFIG,
    registry_db_path=REGISTRY_DB_PATH,
)
logger.info("Evaluation: %s", evaluation_id)


## Register baselines

BM25 is the lexical baseline. Qwen3 uses an explicit biomedical retrieval instruction for queries while documents are encoded without an instruction.

In [ ]:
bm25_pipeline_id = register_pipeline(
    retriever_type="bm25",
    registry_db_path=REGISTRY_DB_PATH,
)

qwen_pipeline_id = register_pipeline(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    similarity_metric="cosine",
    query_prompt=(
        "Instruct: Given a biomedical question, retrieve relevant scientific "
        "passages that answer the question\nQuery: "
    ),
    registry_db_path=REGISTRY_DB_PATH,
)

pipelines = {
    "BM25": bm25_pipeline_id,
    "Qwen3-Embedding-0.6B": qwen_pipeline_id,
}
pipelines


## Evaluate every dataset

Result identity is `(pipeline_id, evaluation_id, dataset_id)`. Changing runtime settings does not create a new result.

In [ ]:
records = []
for pipeline_name, pipeline_id in pipelines.items():
    outcomes = evaluate(
        pipeline_id=pipeline_id,
        evaluation_id=evaluation_id,
        datasets_root=DATASETS_ROOT,
        runtime=runtime,
        registry_db_path=REGISTRY_DB_PATH,
        results_db_path=RESULTS_DB_PATH,
    )
    for outcome in outcomes:
        record = {
            "pipeline": pipeline_name,
            "dataset": outcome.dataset_name,
            "version": outcome.dataset_version,
            "status": outcome.status.value,
            "dataset_id": outcome.dataset_id,
            "result_id": outcome.result_id,
        }
        if outcome.metrics is not None:
            record.update(outcome.metrics)
        records.append(record)

results_table = pd.DataFrame(records).sort_values(
    ["pipeline", "dataset", "version"]
)
identifier_columns = {
    "pipeline", "dataset", "version", "status",
    "dataset_id", "result_id",
}
metric_columns = [
    column for column in results_table.columns
    if column not in identifier_columns
]
display(results_table.style.format(
    {column: "{:.4f}" for column in metric_columns},
    na_rep="",
))
